In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import vstack
from collections import Counter

#Load news dataset with the columns
news_df = pd.read_csv(
    "../MINDsmall_train/news.tsv",
    sep="\t",
    header=None,
    names=[
        "news_id", "category", "subcategory",
        "title", "abstract", "url",
        "title_entities", "abstract_entities"
    ]
)

#Preprocess the data to fill missing values 
news_df["title"]    = news_df["title"].fillna("")
news_df["abstract"] = news_df["abstract"].fillna("")
news_df["url"]      = news_df["url"].fillna("")

#Select required columns
news_df = news_df[["news_id", "category", "subcategory", "title", "abstract", "url"]]

#Combine title and abstract column
news_df["content"] = news_df["title"] + " " + news_df["abstract"]

#Load behaviors with the columns
behaviors_df = pd.read_csv(
    "../MINDsmall_train/behaviors.tsv",
    sep="\t",
    header=None,
    names=["impression_id", "user_id", "time", "history", "impressions"]
)

#Preprocess the data to fill missing values and split history into a list

behaviors_df["history"] = behaviors_df["history"].fillna("")
behaviors_df["history"] = behaviors_df["history"].apply(lambda x: x.split())

#Create a TD-IDF matrix for the news content
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = tfidf.fit_transform(news_df["content"])

#Map NEWS_ID
news_id_to_index = {nid: idx for idx, nid in enumerate(news_df["news_id"])}

#Recency score
news_df["recency_score"] = news_df.index / len(news_df)

print(f"News loaded       : {len(news_df):,} articles")
print(f"Behaviors loaded  : {len(behaviors_df):,} records")
print(f"TF-IDF matrix     : {tfidf_matrix.shape}")

News loaded       : 51,282 articles
Behaviors loaded  : 156,965 records
TF-IDF matrix     : (51282, 5000)


In [3]:
#Existing functions

def build_user_profile(clicked_news_ids):
    """
    Weighted average of TF-IDF vectors for clicked articles.
    More recent clicks receive higher weight: weight = (i+1) / total
    """
    vectors = []
    weights = []
    total   = len(clicked_news_ids)

    for i, news_id in enumerate(clicked_news_ids):
        if news_id in news_id_to_index:
            idx = news_id_to_index[news_id]
            vectors.append(tfidf_matrix[idx])
            weights.append((i + 1) / total)

    if len(vectors) == 0:
        return None

    stacked          = vstack(vectors)
    weights          = np.array(weights).reshape(-1, 1)
    weighted_profile = stacked.multiply(weights).sum(axis=0) / weights.sum()

    return np.asarray(weighted_profile)


def recommend_articles_with_recency(
    user_profile,
    clicked_news_ids,
    selected_category=None,
    selected_subcategory=None,
    alpha=0.9,
    beta=0.1,
    top_n=5,
    candidate_pool=100
):
    """
    Rank articles by: final_score = alpha * similarity + beta * recency
    Scans top `candidate_pool` similarity results, then re-sorts by final_score.
    Filters out clicked articles, and optionally by category / subcategory.
    """
    similarity_scores = cosine_similarity(user_profile, tfidf_matrix).flatten()
    sorted_indices    = similarity_scores.argsort()[::-1]

    recommendations = []

    for idx in sorted_indices:
        row     = news_df.iloc[idx]
        news_id = row["news_id"]

        if news_id in clicked_news_ids:
            continue
        if selected_category    and row["category"]    != selected_category:
            continue
        if selected_subcategory and row["subcategory"] != selected_subcategory:
            continue

        sim_score     = similarity_scores[idx]
        recency_score = row["recency_score"]
        final_score   = alpha * sim_score + beta * recency_score

        recommendations.append((idx, sim_score, recency_score, final_score))

        if len(recommendations) >= candidate_pool:
            break

    recommendations = sorted(recommendations, key=lambda x: x[3], reverse=True)[:top_n]

    results = []
    for idx, sim, rec, final in recommendations:
        row = news_df.iloc[idx][["news_id", "title", "abstract", "category", "subcategory", "url"]].to_dict()
        row["similarity_score"] = round(sim,   4)
        row["recency_score"]    = round(rec,   4)
        row["final_score"]      = round(final, 4)
        results.append(row)

    return pd.DataFrame(results)


def create_user_profile_from_interest(category, subcategory, n_seed=5):
    """
    Cold-start: builds an initial profile from seed articles
    in the chosen category/subcategory.
    """
    seed_articles = news_df[
        (news_df["category"]    == category) &
        (news_df["subcategory"] == subcategory)
    ].head(n_seed)

    clicked_ids  = seed_articles["news_id"].tolist()
    user_profile = build_user_profile(clicked_ids)

    return user_profile, clicked_ids


print("Core functions loaded.")

Core functions loaded.


In [4]:
#Popularity scores. Calculate how many times each articles was cliekced across all behaviours.

def compute_popularity_scores(behaviors_df, news_df):
    """
    Count how many times each article appears as a click (label=1)
    across all impressions in behaviors_df.
    Normalise to [0, 1].
    """
    click_counts = Counter()

    for impression_str in behaviors_df["impressions"]:
        # impressions column is still raw string here
        for item in impression_str.split():
            news_id, label = item.split("-")
            if int(label) == 1:
                click_counts[news_id] += 1

    # Map counts onto news_df
    news_df["popularity_raw"]   = news_df["news_id"].map(click_counts).fillna(0)
    max_clicks                  = news_df["popularity_raw"].max()
    news_df["popularity_score"] = news_df["popularity_raw"] / max_clicks if max_clicks > 0 else 0

    return news_df


# Load raw behaviors for popularity counting (impressions still as strings)
behaviors_raw = pd.read_csv(
    "../MINDsmall_train/behaviors.tsv",
    sep="\t",
    header=None,
    names=["impression_id", "user_id", "time", "history", "impressions"]
)

news_df = compute_popularity_scores(behaviors_raw, news_df)

print(f"Popularity scores computed.")
print(f"Most clicked article : {int(news_df['popularity_raw'].max())} clicks")
print(f"Articles with 0 clicks: {(news_df['popularity_raw'] == 0).sum():,}")
news_df[["news_id", "popularity_raw", "popularity_score"]].sort_values(
    "popularity_raw", ascending=False
).head(5)

Popularity scores computed.
Most clicked article : 4316 clicks
Articles with 0 clicks: 43,569


,news_id,popularity_raw,popularity_score
33899,N55689,4316,1.000000
50688,N35729,3346,0.775255
41550,N33619,3246,0.752085
45974,N53585,2835,0.656858
41552,N63970,2578,0.597312


In [5]:
# UserSession Class
# Holds all live state for one user session.


class UserSession:
    """
    Manages a single user's live recommendation session.

    Attributes
    ----------
    clicked_ids        : list  — all article IDs the user has clicked
    user_profile       : np.array — current TF-IDF profile vector
    selected_category  : str  — category chosen at cold start
    selected_subcategory: str — subcategory chosen at cold start
    round_number       : int  — current interaction round
    session_log        : list — history of every round's shown/clicked articles
    """

    def __init__(self, category, subcategory):
        self.selected_category    = category
        self.selected_subcategory = subcategory
        self.round_number         = 0
        self.session_log          = []

        # Cold start: seed profile from chosen category/subcategory
        self.user_profile, self.clicked_ids = create_user_profile_from_interest(
            category, subcategory
        )

        print(f"\n Session started!")
        print(f" Category    : {category}")
        print(f" Subcategory : {subcategory}")
        print(f" Seed articles (auto-clicked to build initial profile): {len(self.clicked_ids)}")


    def update(self, clicked_news_id):
        """
        Call after user clicks an article:
        1. Add article to click history
        2. Rebuild user profile with updated history
        """
        if clicked_news_id not in self.clicked_ids:
            self.clicked_ids.append(clicked_news_id)

        # Rebuild profile — uses the full click history every time
        self.user_profile = build_user_profile(self.clicked_ids)


    def get_dominant_category(self):
        """
        Returns the most common category among clicked articles.
        Shows how the user's interest has evolved.
        """
        clicked_rows = news_df[news_df["news_id"].isin(self.clicked_ids)]
        if clicked_rows.empty:
            return self.selected_category
        return clicked_rows["category"].value_counts().idxmax()


    def get_interest_summary(self):
        """
        Returns a breakdown of categories read so far.
        """
        clicked_rows = news_df[news_df["news_id"].isin(self.clicked_ids)]
        return clicked_rows["category"].value_counts().to_dict()


    def log_round(self, shown_ids, clicked_id):
        self.session_log.append({
            "round"      : self.round_number,
            "shown"      : shown_ids,
            "clicked"    : clicked_id,
            "profile_cat": self.get_dominant_category()
        })

print("UserSession class ready.")

UserSession class ready.


In [6]:
#Enhanced Recommender with Popularity + Diversity

def recommend_with_popularity(
    user_profile,
    clicked_news_ids,
    selected_category=None,
    selected_subcategory=None,
    alpha=0.8,      # similarity weight
    beta=0.1,       # recency weight
    gamma=0.1,      # popularity weight
    top_n=5,
    candidate_pool=150
):
    """
    Extended scorer:
        final_score = alpha * similarity + beta * recency + gamma * popularity

    Scans top `candidate_pool` similarity candidates, then re-sorts.
    """
    similarity_scores = cosine_similarity(user_profile, tfidf_matrix).flatten()
    sorted_indices    = similarity_scores.argsort()[::-1]

    candidates = []

    for idx in sorted_indices:
        row     = news_df.iloc[idx]
        news_id = row["news_id"]

        if news_id in clicked_news_ids:
            continue
        if selected_category    and row["category"]    != selected_category:
            continue
        if selected_subcategory and row["subcategory"] != selected_subcategory:
            continue

        sim_score        = similarity_scores[idx]
        recency_score    = row["recency_score"]
        popularity_score = row["popularity_score"]
        final_score      = alpha * sim_score + beta * recency_score + gamma * popularity_score

        candidates.append((idx, sim_score, recency_score, popularity_score, final_score))

        if len(candidates) >= candidate_pool:
            break

    candidates = sorted(candidates, key=lambda x: x[4], reverse=True)

    results = []
    for idx, sim, rec, pop, final in candidates[:top_n]:
        row = news_df.iloc[idx][["news_id", "title", "abstract", "category", "subcategory", "url"]].to_dict()
        row["similarity_score"]  = round(sim,   4)
        row["recency_score"]     = round(rec,   4)
        row["popularity_score"]  = round(pop,   4)
        row["final_score"]       = round(final, 4)
        results.append(row)

    return pd.DataFrame(results)


def inject_diversity(recommendations_df, clicked_news_ids, n_diverse=1):
    """
    Replaces the last `n_diverse` slots in the recommendation list
    with articles from a DIFFERENT subcategory than the top result.

    Prevents the recommendations from converging into an echo chamber.

    Parameters
    ----------
    recommendations_df : DataFrame  — current top-N recommendations
    clicked_news_ids   : list       — already-seen articles to exclude
    n_diverse          : int        — how many diversity slots to inject (default 1)
    """
    if recommendations_df.empty:
        return recommendations_df

    dominant_subcategory = recommendations_df.iloc[0]["subcategory"]
    shown_ids            = set(recommendations_df["news_id"].tolist())
    exclude_ids          = set(clicked_news_ids) | shown_ids

    # Find articles from a different subcategory
    diverse_pool = news_df[
        (news_df["subcategory"] != dominant_subcategory) &
        (~news_df["news_id"].isin(exclude_ids))
    ].sort_values("popularity_score", ascending=False)

    if diverse_pool.empty:
        return recommendations_df  # no diverse articles available, return as-is

    diverse_articles = diverse_pool.head(n_diverse).copy()
    diverse_articles["similarity_score"] = 0.0
    diverse_articles["recency_score"]    = diverse_articles["recency_score"]
    diverse_articles["popularity_score"] = diverse_articles["popularity_score"]
    diverse_articles["final_score"]      = 0.0
    diverse_articles["is_diverse"]       = True

    # Keep all columns aligned
    keep_cols = ["news_id", "title", "abstract", "category", "subcategory",
                 "url", "similarity_score", "recency_score", "popularity_score", "final_score"]

    recommendations_df = recommendations_df.copy()
    recommendations_df["is_diverse"] = False

    # Replace last n_diverse slots
    main_recs = recommendations_df.iloc[:len(recommendations_df) - n_diverse]

    combined = pd.concat([main_recs, diverse_articles[[*keep_cols]].assign(is_diverse=True)],
                         ignore_index=True)

    return combined


print("Enhanced recommender and diversity injection ready.")

Enhanced recommender and diversity injection ready.


In [7]:
#Display Helpers

def display_recommendations(recommendations_df, round_number):
    """
    Prints the recommendations in a clean, readable format.
    Marks the diversity-injected article clearly.
    """
    print(f"\n{'='*65}")
    print(f"  ROUND {round_number}  —  Top Recommendations")
    print(f"{'='*65}")

    for i, row in recommendations_df.iterrows():
        display_idx  = i + 1
        is_diverse   = row.get("is_diverse", False)
        diverse_tag  = "  [EXPLORE]" if is_diverse else ""

        title_short  = row["title"][:70] + "..." if len(row["title"]) > 70 else row["title"]
        abstract_short = row["abstract"][:90] + "..." if len(str(row["abstract"])) > 90 else row["abstract"]

        print(f"\n  [{display_idx}] {title_short}{diverse_tag}")
        print(f"       Category : {row['category']} > {row['subcategory']}")
        print(f"       Score    : {row['final_score']:.4f}  "
              f"(sim={row['similarity_score']:.3f}, "
              f"rec={row['recency_score']:.3f}, "
              f"pop={row['popularity_score']:.3f})")
        print(f"       Preview  : {abstract_short}")

    print(f"\n{'─'*65}")


def display_profile_drift(session):
    """
    After each click, shows how the user's interest profile has evolved.
    """
    summary = session.get_interest_summary()
    dominant = session.get_dominant_category()

    print(f"\n  Profile update:")
    print(f"  Total articles in history : {len(session.clicked_ids)}")
    print(f"  Dominant interest         : {dominant}")
    print(f"  Interest breakdown        :", end=" ")

    for cat, count in sorted(summary.items(), key=lambda x: x[1], reverse=True):
        bar = "█" * count
        print(f"\n    {cat:<20} {bar} ({count})", end="")

    print()


def display_session_summary(session):
    """
    Printed when the user quits. Full session recap.
    """
    print(f"\n{'='*65}")
    print(f"  SESSION SUMMARY")
    print(f"{'='*65}")
    print(f"  Rounds completed     : {session.round_number}")
    print(f"  Total articles read  : {len(session.clicked_ids)}")
    print(f"  Started with         : {session.selected_category} > {session.selected_subcategory}")
    print(f"  Final dominant topic : {session.get_dominant_category()}")

    print(f"\n  Interest evolution by round:")
    for entry in session.session_log:
        clicked_title = ""
        clicked_rows  = news_df[news_df["news_id"] == entry["clicked"]]
        if not clicked_rows.empty:
            t = clicked_rows.iloc[0]["title"]
            clicked_title = t[:55] + "..." if len(t) > 55 else t
        print(f"  Round {entry['round']:>2}  |  Clicked: {clicked_title}")
        print(f"           |  Profile dominant: {entry['profile_cat']}")

    print(f"\n  Categories explored:")
    for cat, count in sorted(session.get_interest_summary().items(),
                              key=lambda x: x[1], reverse=True):
        print(f"    {cat:<25} {count} articles")

    print(f"\n{'='*65}")
    print("  Thanks for using the News Recommender!")
    print(f"{'='*65}\n")


print("Display helpers ready.")

Display helpers ready.


In [8]:
#Run_session(): The Main Interaction Loop

def run_session(
    top_n=5,
    diversity_slots=1,
    alpha=0.8,
    beta=0.1,
    gamma=0.1,
    use_popularity=True,
    filter_by_category=True
):
    """
    Full interactive recommendation session.

    Parameters
    ----------
    top_n            : number of articles to recommend each round
    diversity_slots  : how many slots to give to diversity injection
    alpha            : similarity weight
    beta             : recency weight
    gamma            : popularity weight (only used if use_popularity=True)
    use_popularity   : whether to include popularity in scoring
    filter_by_category: whether to stay within selected category
    """

    # ── COLD START: Category selection ─────────────────────
    print("\n" + "="*65)
    print("  WELCOME TO THE INTERACTIVE NEWS RECOMMENDER")
    print("="*65)
    print("\nAvailable Categories:\n")

    categories = sorted(news_df["category"].unique())
    for i, cat in enumerate(categories):
        print(f"  {i:>2}: {cat}")

    cat_index         = int(input("\nSelect category number: "))
    selected_category = categories[cat_index]

    # ── COLD START: Subcategory selection ──────────────────
    subcats = sorted(
        news_df[news_df["category"] == selected_category]["subcategory"].unique()
    )

    print(f"\nAvailable Subcategories for '{selected_category}':\n")
    for i, sub in enumerate(subcats):
        print(f"  {i:>2}: {sub}")

    sub_index            = int(input("\nSelect subcategory number: "))
    selected_subcategory = subcats[sub_index]

    # ── Initialise session ──────────────────────────────────
    session = UserSession(selected_category, selected_subcategory)

    cat_filter = selected_category if filter_by_category else None

    # ── MAIN LOOP ───────────────────────────────────────────
    while True:
        session.round_number += 1

        # Generate recommendations
        if use_popularity:
            recs = recommend_with_popularity(
                session.user_profile,
                session.clicked_ids,
                selected_category=cat_filter,
                alpha=alpha, beta=beta, gamma=gamma,
                top_n=top_n
            )
        else:
            recs = recommend_articles_with_recency(
                session.user_profile,
                session.clicked_ids,
                selected_category=cat_filter,
                alpha=alpha, beta=beta,
                top_n=top_n
            )

        # Fallback: if filtered pool is exhausted, open up to all categories
        if recs.empty:
            print("\n  Not enough articles in this category. Expanding to all categories...")
            cat_filter = None
            recs = recommend_with_popularity(
                session.user_profile,
                session.clicked_ids,
                alpha=alpha, beta=beta, gamma=gamma,
                top_n=top_n
            )

        # Inject diversity
        if diversity_slots > 0 and len(recs) >= diversity_slots + 1:
            recs = inject_diversity(recs, session.clicked_ids, n_diverse=diversity_slots)

        # Display
        display_recommendations(recs, session.round_number)

        # ── User input ──────────────────────────────────────
        print(f"  Enter 1–{len(recs)} to read an article")
        print(f"  Enter 'q' to quit")
        user_input = input("\n  Your choice: ").strip().lower()

        if user_input == "q":
            display_session_summary(session)
            break

        # Validate input
        try:
            choice = int(user_input)
            if choice < 1 or choice > len(recs):
                print(f"\n  Please enter a number between 1 and {len(recs)}.")
                session.round_number -= 1  # don't count invalid round
                continue
        except ValueError:
            print("\n  Invalid input. Please enter a number or 'q'.")
            session.round_number -= 1
            continue

        # ── Process click ───────────────────────────────────
        clicked_row     = recs.iloc[choice - 1]
        clicked_news_id = clicked_row["news_id"]
        clicked_title   = clicked_row["title"]

        print(f"\n  You selected: {clicked_title[:70]}")
        print(f"  Category    : {clicked_row['category']} > {clicked_row['subcategory']}")

        # Log round BEFORE updating profile
        session.log_round(
            shown_ids=recs["news_id"].tolist(),
            clicked_id=clicked_news_id
        )

        # Update session: add click, rebuild profile
        session.update(clicked_news_id)

        # Show profile drift
        display_profile_drift(session)


print("run_session() ready. Run the next cell to start.")

run_session() ready. Run the next cell to start.


In [9]:
#Launch the Session

run_session(
    top_n=5,               # articles shown per round
    diversity_slots=1,     # 1 out of 5 slots is a diversity pick
    alpha=0.8,             # similarity weight
    beta=0.1,              # recency weight
    gamma=0.1,             # popularity weight
    use_popularity=True,   # set False to revert to original scoring
    filter_by_category=True  # set False to recommend across all categories
)


  WELCOME TO THE INTERACTIVE NEWS RECOMMENDER

Available Categories:

   0: autos
   1: entertainment
   2: finance
   3: foodanddrink
   4: health
   5: kids
   6: lifestyle
   7: middleeast
   8: movies
   9: music
  10: news
  11: northamerica
  12: sports
  13: travel
  14: tv
  15: video
  16: weather

Available Subcategories for 'travel':

   0: holidays
   1: internationaltravel
   2: newstrends
   3: travel-adventure-travel
   4: travel-points-rewards
   5: travel-videos
   6: travelarticle
   7: travelnews
   8: traveltips
   9: traveltripideas
  10: traveltrivia
  11: ustravel
  12: video
  13: voices

 Session started!
 Category    : travel
 Subcategory : holidays
 Seed articles (auto-clicked to build initial profile): 2

  ROUND 1  —  Top Recommendations

  [1] The Best Winter Vacation Destinations for a Wow-Worthy Getaway
       Category : travel > traveltripideas
       Score    : 0.3784  (sim=0.397, rec=0.612, pop=0.000)
       Preview  : From tropical beach getaways to